In [108]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import pickle
import copy
import os
import gc
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

In [109]:
df = pd.read_csv('question/data.csv')

In [110]:
df

,age,fnlwgt,educational-num,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,...,occupation_Protective-serv,occupation_Sales,occupation_Tech-support,occupation_Transport-moving,relationship_Husband,relationship_Not-in-family,relationship_Other-relative,relationship_Own-child,relationship_Unmarried,relationship_Wife
0,25,226802,7,0,1,0,0,40,0,0,...,False,False,False,False,False,False,False,True,False,False
1,38,89814,9,1,1,0,0,50,0,0,...,False,False,False,False,True,False,False,False,False,False
2,28,336951,12,1,1,0,0,40,0,1,...,True,False,False,False,True,False,False,False,False,False
3,44,160323,10,0,1,7688,0,40,0,1,...,False,False,False,False,True,False,False,False,False,False
4,18,103497,10,1,0,0,0,30,0,0,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,257302,12,1,0,0,0,38,0,0,...,False,False,True,False,False,False,False,False,False,True
48838,40,154374,9,1,1,0,0,40,0,1,...,False,False,False,False,True,False,False,False,False,False
48839,58,151910,9,1,0,0,0,40,0,0,...,False,False,False,False,False,False,False,False,True,False
48840,22,201490,9,1,1,0,0,20,0,0,...,False,False,False,False,False,False,False,True,False,False


# Part 1

In [111]:
def zemel_fairness(dataframe, pred_income):
    total_women = dataframe[dataframe['gender'] == 0].shape[0]
    total_men = dataframe[dataframe['gender'] == 1].shape[0]
    prob_pos_given_women = dataframe[(dataframe['gender'] == 0) & (pred_income == 1)].shape[0] / total_women 
    prob_pos_given_men = dataframe[(dataframe['gender'] == 1) & (pred_income == 1)].shape[0] / total_men
    return prob_pos_given_men - prob_pos_given_women

def disparate_impact(dataframe, pred_income):
    total_women = dataframe[dataframe['gender'] == 0].shape[0]
    total_men = dataframe[dataframe['gender'] == 1].shape[0]
    prob_pos_given_women = dataframe[(dataframe['gender'] == 0) & (pred_income == 1)].shape[0] / total_women 
    prob_pos_given_men = dataframe[(dataframe['gender'] == 1) & (pred_income == 1)].shape[0] / total_men
    return prob_pos_given_women / prob_pos_given_men

def fairness_eval(dataframe, pred_income):
    total_women = dataframe[dataframe['gender'] == 0].shape[0]
    total_men = dataframe[dataframe['gender'] == 1].shape[0]
    prob_pos_given_women = dataframe[(dataframe['gender'] == 0) & (pred_income == 1)].shape[0] / total_women 
    prob_pos_given_men = dataframe[(dataframe['gender'] == 1) & (pred_income == 1)].shape[0] / total_men
    print('probability of income >50k for women =', prob_pos_given_women)
    print('probability of income >50k for men =', prob_pos_given_men)
    zemel_fairness_value = zemel_fairness(dataframe, pred_income)
    print('Zemel Fairness =', zemel_fairness_value)
    disparate_impact_value = disparate_impact(dataframe, pred_income)
    print('Disparate Impact =', disparate_impact_value)

In [112]:
# total_women = dataframe[dataframe['gender'] == 0].shape[0]
# total_men = dataframe[dataframe['gender'] == 1].shape[0]
# prob_pos_given_women = dataframe[(dataframe['gender'] == 0) & (income = 1)].shape[0] / total_women 
# prob_pos_given_men = dataframe[(dataframe['gender'] == 1) & (income = 1)].shape[0] / total_men

# Part 2

In [126]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['income'])
x_train, y_train = train_df.drop(columns=['income']), train_df['income']
x_test, y_test = test_df.drop(columns=['income']), test_df['income']
print("Training Set shape:")
print(train_df.shape)
print("\nTest Set shape:")
print(test_df.shape)

Training Set shape:
(34189, 43)

Test Set shape:
(14653, 43)


In [127]:
scaler = StandardScaler()
x_train_normal = scaler.fit_transform(x_train)

# Train the Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
# clf = SVC(random_state=42)
clf.fit(x_train_normal, y_train)

# Predict the test set
x_test_normal = scaler.transform(x_test)
y_pred = clf.predict(x_test_normal)

In [128]:
report = classification_report(y_true=y_test, y_pred=y_pred)
print("Classification Report:")
print(report)
fairness_eval(dataframe=x_test, pred_income=y_pred)

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     11147
           1       0.75      0.62      0.67      3506

    accuracy                           0.86     14653
   macro avg       0.82      0.78      0.79     14653
weighted avg       0.85      0.86      0.85     14653

probability of income >50k for women = 0.08112003294214536
probability of income >50k for men = 0.25541037158023683
Zemel Fairness = 0.17429033863809146
Disparate Impact = 0.3176066517590951


## Remove sensitive attribute

In [129]:
train_df.columns

Index(['age', 'fnlwgt', 'educational-num', 'race', 'gender', 'capital-gain',
       'capital-loss', 'hours-per-week', 'native-country', 'income',
       'workclass_Federal-gov', 'workclass_Local-gov',
       'workclass_Never-worked', 'workclass_Private', 'workclass_Self-emp-inc',
       'workclass_Self-emp-not-inc', 'workclass_State-gov',
       'workclass_Without-pay', 'marital_status_divorced',
       'marital_status_married', 'marital_status_separated',
       'marital_status_unmarried', 'marital_status_widowed',
       'occupation_Adm-clerical', 'occupation_Armed-Forces',
       'occupation_Craft-repair', 'occupation_Exec-managerial',
       'occupation_Farming-fishing', 'occupation_Handlers-cleaners',
       'occupation_Machine-op-inspct', 'occupation_Other-service',
       'occupation_Priv-house-serv', 'occupation_Prof-specialty',
       'occupation_Protective-serv', 'occupation_Sales',
       'occupation_Tech-support', 'occupation_Transport-moving',
       'relationship_Husband'

In [130]:
x_train_without_gender, y_train = train_df.drop(columns=['income', 'gender']), train_df['income']
x_test_without_gender, y_test = test_df.drop(columns=['income', 'gender']), test_df['income']

scaler = StandardScaler()
x_train_normal_without_gender = scaler.fit_transform(x_train_without_gender)
# Train the Random Forest classifier
clf_without_gender = RandomForestClassifier(n_estimators=100, random_state=42)
# clf_without_gender = SVC(random_state=42)
clf_without_gender.fit(x_train_normal_without_gender, y_train)

# Predict the test set
x_test_normal_without_gender = scaler.transform(x_test_without_gender)
y_pred_without_gender = clf_without_gender.predict(x_test_normal_without_gender)

In [131]:
report = classification_report(y_true=y_test, y_pred=y_pred_without_gender)
print("Classification Report:")
print(report)
fairness_eval(dataframe=x_test, pred_income=y_pred_without_gender)

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     11147
           1       0.74      0.63      0.68      3506

    accuracy                           0.86     14653
   macro avg       0.82      0.78      0.79     14653
weighted avg       0.85      0.86      0.85     14653

probability of income >50k for women = 0.08585546633724521
probability of income >50k for men = 0.25796243364638627
Zemel Fairness = 0.17210696730914105
Disparate Impact = 0.33282158616527663


In [132]:
# no significant impact

# part 3

In [133]:
new_train_df = pd.DataFrame(train_df)

new_train_df['prediction'] = clf.predict(x_train.to_numpy())

new_train_df['max_prob'] = clf.predict_proba(x_train.to_numpy()).max(1)

In [134]:
# new_df = pd.DataFrame(df)

# new_df['prediction'] = clf.predict(df.drop(columns=['income']))

# new_df['max_prob'] = clf.predict_proba(df.drop(columns=['income'])).max(1)

In [135]:
_cp = new_train_df[(new_train_df['gender'] == 1) & (y_train == 1)]
_cd = new_train_df[(new_train_df['gender'] == 0) & (y_train == 0)]

cp = _cp.sort_values(by='max_prob', ascending=True)
cd = _cd.sort_values(by='max_prob', ascending=False)

In [136]:
cp.shape, cd.shape

((6965, 45), (10119, 45))

In [137]:
# Calculate Ss, S's, S+s, S'+s
Ss = sum((new_train_df['gender'] == 0))
Ss_bar = sum((new_train_df['gender'] == 1))
S_s_plus = sum((new_train_df['gender'] == 0) & (new_train_df['prediction'] == 1))
S_s_bar_plus = sum((new_train_df['gender'] == 1) & (new_train_df['prediction'] == 1))

# Calculate n using the provided formula
n = int((Ss * S_s_bar_plus - Ss_bar * S_s_plus) / (Ss + Ss_bar))

In [138]:
# n = 1000

In [139]:
n_cp_indices = cp.iloc[:n].copy().index
n_cd_indices = cd.iloc[:n].copy().index

In [140]:
n

-218

In [122]:
new_train_df.loc[n_cp_indices, 'income'], new_train_df.loc[n_cd_indices, 'income'] = list(new_train_df.loc[n_cd_indices, 'income']), list(new_train_df.loc[n_cp_indices, 'income'])

In [123]:
new_train_df = new_train_df.drop(columns=['prediction', 'max_prob'])

In [124]:
new_x_train, new_y_train = new_train_df.drop(columns=['income']), new_train_df['income']
x_test, y_test = test_df.drop(columns=['income']), test_df['income']
print("Training Set shape:")
print(train_df.shape)
print("\nTest Set shape:")
print(test_df.shape)

scaler = StandardScaler()
new_x_train_normal = scaler.fit_transform(new_x_train)

# Train the Random Forest classifier
clf_part3 = RandomForestClassifier(n_estimators=100, random_state=42)
clf_part3.fit(new_x_train_normal, new_y_train)

# Predict the test set
new_x_test_normal = scaler.transform(x_test)
y_pred_part3 = clf_part3.predict(new_x_test_normal)

Training Set shape:
(34189, 43)

Test Set shape:
(14653, 43)


In [125]:
report = classification_report(y_true=y_test, y_pred=y_pred_part3)
print("Classification Report:")
print(report)
fairness_eval(dataframe=x_test, pred_income=y_pred_part3)

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     11233
           1       0.70      0.57      0.63      3420

    accuracy                           0.84     14653
   macro avg       0.79      0.75      0.76     14653
weighted avg       0.83      0.84      0.84     14653

probability of income >50k for women = 0.13090084518655948
probability of income >50k for men = 0.22056723117731075
Zemel Fairness = 0.08966638599075127
Disparate Impact = 0.5934736746154745


# part 5

## method 1

In [24]:
df = pd.read_csv('question/data.csv')

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42)

print("Training Set shape:")
print(train_df.shape)
print("\nTest Set shape:")
print(test_df.shape)

Training Set shape:
(34189, 43)

Test Set shape:
(14653, 43)


In [25]:
# Identify counts of men and women
count_female = train_df[train_df['gender'] == 0].shape[0]
count_male = train_df[train_df['gender'] == 1].shape[0]
print('Before balancing the data:')
print('number of female:', count_female)
print('number of male:', count_male)
# Determine the underrepresented group
if count_female < count_male:
    minority_class = 0  # Female
    majority_class = 1  # Male
    diff = count_male - count_female
else:
    minority_class = 1  # Male
    majority_class = 0  # Female
    diff = count_female - count_male

# Get minority group data
minority_data = train_df[train_df['gender'] == minority_class]

# Repeat data points until counts are equal
while diff > 0:
    if diff >= minority_data.shape[0]:
        train_df = pd.concat([train_df, minority_data])
        diff -= minority_data.shape[0]
    else:
        train_df = pd.concat([train_df, minority_data.sample(diff)])
        diff = 0

count_female = train_df[train_df['gender'] == 0].shape[0]
count_male = train_df[train_df['gender'] == 1].shape[0]
print('\nAfter balancing the data:')
print('number of female:', count_female)
print('number of male:', count_male)

Before balancing the data:
number of female: 11341
number of male: 22848

After balancing the data:
number of female: 22848
number of male: 22848


In [26]:
x_train, y_train = train_df.drop(columns=['income']), train_df['income']
x_test, y_test = test_df.drop(columns=['income']), test_df['income']

scaler = StandardScaler()
x_train_normal = scaler.fit_transform(x_train)

# Train the Random Forest classifier
clf = MLPClassifier(hidden_layer_sizes=(128,32),max_iter=500 , random_state=42)
clf.fit(x_train_normal, y_train)

# Predict the test set
x_test_normal = scaler.transform(x_test)
y_pred = clf.predict(x_test_normal)

In [27]:
report = classification_report(y_true=y_test, y_pred=y_pred)
print("Classification Report:")
print(report)
fairness_eval(dataframe=x_test, pred_income=y_pred)

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.89      0.89     11233
           1       0.64      0.61      0.63      3420

    accuracy                           0.83     14653
   macro avg       0.76      0.75      0.76     14653
weighted avg       0.83      0.83      0.83     14653

probability of income >50k for women = 0.10039167182024325
probability of income >50k for men = 0.28504386859824526
Zemel Fairness = 0.18465219677800201
Disparate Impact = 0.3521972681395935


## method 2
proposed in file of part5